# inference.ipynb — 최종 추론 및 제출 파일 생성

**목적**: `train.ipynb`에서 저장한 모델(`models/catboost_seed{42,7,2024}.cbm`, `models/mlp_seed{42,7,2024}.pt`)과 설정(`models/model_config.json`)을 불러와 2025년 테스트 기간을 예측하고, 대회 제출 규격(`src/submission.py`)에 맞는 CSV를 만든다. **외부 API 호출 없음, 이 노트북만 단독으로 처음부터 끝까지 실행 가능**(2차 평가 요건 — `ensemble-final` 스킬 5절).

**최종 모델 구조**(`05_tuning.ipynb` 14절 확정): CatBoost(τ=0.70+actual가중) seed 평균과 MLP(`soft_metric_loss`, T_soft=0.003) seed 평균을 그룹별 가중치로 블렌드한다(group_1=0.4/group_2=0.5/group_3=1.0 — group_3은 MLP만 사용). 그룹별 가중치는 `model_config.json`의 `blend_weights`에서 읽는다.

**입력**: `data/processed/test_features_v2.parquet`, `models/catboost_seed*.cbm`, `models/mlp_seed*.pt`, `models/mlp_scaler.npz`, `models/model_config.json`, `data/sample_submission.csv`
**출력**: `submissions/YYYYMMDD_vN_설명.csv`

**주의**: 이 노트북은 로컬에 제출 파일(CSV)만 만든다. 실제 대회 플랫폼에 제출(업로드)하는 것은 별도 단계이며 민석님이 직접 하고, 일일 제출 5회 제한을 고려해 신중하게 선택한다.

## 0. 셋업

In [1]:
import sys
import json
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostRegressor

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import TARGET_COLS, CAPACITY_KWH
from src.submission import validate_submission
from src import nn as mlp_nn

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"
SUBMISSIONS_DIR = REPO_ROOT / "submissions"
SUBMISSIONS_DIR.mkdir(exist_ok=True)

print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)
print("torch:", torch.__version__)

python executable: d:\공모전\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: d:\공모전\wind_forecast
torch: 2.13.0+cpu


## 1. 설정·모델 로드

`train.ipynb`가 저장한 `model_config.json`을 그대로 읽어 피처 목록·시드·손실 함수·MLP 설정·블렌드 가중치를 복원한다. 값을 이 노트북에 다시 하드코딩하지 않는 이유: 두 노트북 사이에 설정이 어긋나면(예: 피처 목록이 조금이라도 다르면) 조용히 틀린 예측이 나올 수 있기 때문 — 파일로 한 번만 정의하고 양쪽이 그 파일을 참조하는 게 안전하다. CatBoost와 MLP 둘 다 불러온다.

In [2]:
with open(MODELS_DIR / "model_config.json", encoding="utf-8") as f:
    model_config = json.load(f)

FEATURE_COLS = model_config["feature_cols"]
GROUP_ID_MAP = model_config["group_id_map"]
ENSEMBLE_SEEDS = model_config["ensemble_seeds"]
GROUP_ID_CATEGORIES = list(GROUP_ID_MAP.values())
BLEND_WEIGHTS = model_config["blend_weights"]

MLP_T_SOFT = model_config["mlp"]["t_soft"]
MLP_HIDDEN = tuple(model_config["mlp"]["hidden"])
MLP_DROPOUT = model_config["mlp"]["dropout"]
MLP_INPUT_DIM = model_config["mlp"]["input_dim"]

models = {}
for seed in ENSEMBLE_SEEDS:
    model = CatBoostRegressor()
    model.load_model(str(MODELS_DIR / f"catboost_seed{seed}.cbm"))
    models[seed] = model

mlp_scaler = np.load(MODELS_DIR / "mlp_scaler.npz")
mlp_mu, mlp_sd = mlp_scaler["mu"], mlp_scaler["sd"]

mlp_models = {}
for seed in ENSEMBLE_SEEDS:
    mlp_model = mlp_nn.MLP(MLP_INPUT_DIM, hidden=MLP_HIDDEN, dropout=MLP_DROPOUT)
    mlp_model.load_state_dict(torch.load(MODELS_DIR / f"mlp_seed{seed}.pt", map_location="cpu"))
    mlp_model.eval()
    mlp_models[seed] = mlp_model

print("불러온 CatBoost 시드:", list(models.keys()), "| MLP 시드:", list(mlp_models.keys()))
print("피처 개수:", len(FEATURE_COLS), "| MLP 입력 차원:", MLP_INPUT_DIM)
print("quantile_alpha:", model_config["quantile_alpha"], "| use_sample_weight:", model_config["use_sample_weight"])
print("blend_weights:", BLEND_WEIGHTS)

불러온 CatBoost 시드: [42, 7, 2024] | MLP 시드: [42, 7, 2024]
피처 개수: 50 | MLP 입력 차원: 53
quantile_alpha: 0.7 | use_sample_weight: True
blend_weights: {'kpx_group_1': 0.4, 'kpx_group_2': 0.5, 'kpx_group_3': 1.0}


## 2. 테스트 데이터 로드

In [3]:
test_features = pd.read_parquet(PROCESSED_DIR / "test_features_v2.parquet")
print("test_features:", test_features.shape)
print("기간:", test_features["forecast_kst_dtm"].min(), "~", test_features["forecast_kst_dtm"].max())

missing_cols = [c for c in FEATURE_COLS if c not in test_features.columns]
assert not missing_cols, f"train에는 있는데 test에는 없는 피처: {missing_cols}"
print("train.ipynb가 쓴 피처가 test에 모두 존재함을 확인")

test_features: (8760, 52)
기간: 2025-01-01 01:00:00 ~ 2026-01-01 00:00:00
train.ipynb가 쓴 피처가 test에 모두 존재함을 확인


## 3. 그룹별 예측 — CatBoost·MLP 각각 seed 평균 앙상블 후 그룹별 블렌드

`train.ipynb`와 동일하게 그룹마다 `group_id`를 고정해 CatBoost·MLP 각각 예측한 뒤, 이용률→kWh로 환산한다. 두 모델의 seed 평균을 그룹별 블렌드 가중치(`BLEND_WEIGHTS`, `05_tuning` 14-8/14-9에서 seed 3개로 검증)로 섞고 마지막에 `[0, 설비용량]`으로 클리핑한다(먼저 클리핑한 뒤 섞으면 안 됨 — 블렌드 자체가 선형결합이라 순서를 바꾸면 클리핑 경계에서 값이 달라질 수 있다).

In [4]:
features = FEATURE_COLS + ["group_id"]
predictions = {}
cat_predictions = {}
mlp_predictions = {}

for g in TARGET_COLS:
    cat_rows = test_features.copy()
    cat_rows["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(cat_rows), categories=GROUP_ID_CATEGORIES)
    cat_seed_preds = [models[seed].predict(cat_rows[features]) * CAPACITY_KWH[g] for seed in ENSEMBLE_SEEDS]
    cat_pred_g = np.mean(cat_seed_preds, axis=0)

    mlp_rows = test_features.copy()
    mlp_rows["group_id"] = GROUP_ID_MAP[g]
    mlp_X_g, _, _ = mlp_nn.build_features(mlp_rows, FEATURE_COLS, mu=mlp_mu, sd=mlp_sd)
    mlp_seed_preds = [
        mlp_nn.predict_mlp(mlp_models[seed], mlp_X_g) * CAPACITY_KWH[g]
        for seed in ENSEMBLE_SEEDS
    ]
    mlp_pred_g = np.mean(mlp_seed_preds, axis=0)

    w = BLEND_WEIGHTS[g]
    blended = (1 - w) * cat_pred_g + w * mlp_pred_g

    cat_predictions[g] = cat_pred_g
    mlp_predictions[g] = mlp_pred_g
    predictions[g] = np.clip(blended, 0, CAPACITY_KWH[g])

    print(
        f"{g} (w_mlp={w}): CatBoost 범위 [{cat_pred_g.min():.1f}, {cat_pred_g.max():.1f}], "
        f"MLP 범위 [{mlp_pred_g.min():.1f}, {mlp_pred_g.max():.1f}], "
        f"블렌드 클리핑 후 [{predictions[g].min():.1f}, {predictions[g].max():.1f}]"
    )

kpx_group_1 (w_mlp=0.4): CatBoost 범위 [-388.4, 22020.5], MLP 범위 [1101.5, 21033.5], 블렌드 클리핑 후 [377.3, 21240.3]
kpx_group_2 (w_mlp=0.5): CatBoost 범위 [-361.2, 22033.0], MLP 범위 [1039.5, 21140.0], 블렌드 클리핑 후 [478.5, 21203.6]
kpx_group_3 (w_mlp=1.0): CatBoost 범위 [-836.8, 21365.7], MLP 범위 [771.6, 20384.2], 블렌드 클리핑 후 [771.6, 20384.2]


## 4. 제출 형식으로 조립

`sample_submission.csv`의 행 순서(`forecast_id`)를 기준으로 병합해서, 순서가 어긋날 가능성을 원천 차단한다(`src/submission.py`가 순서까지 검사함).

In [5]:
sample_submission = pd.read_csv(
    REPO_ROOT / "data" / "sample_submission.csv",
    encoding="utf-8-sig",
    dtype={"forecast_id": str},
    parse_dates=["forecast_kst_dtm"],
)

pred_df = test_features[["forecast_id", "forecast_kst_dtm"]].copy()
for g in TARGET_COLS:
    pred_df[g] = predictions[g]

submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
    pred_df, on=["forecast_id", "forecast_kst_dtm"], how="left",
)

assert submission[TARGET_COLS].isna().sum().sum() == 0, "병합 후 결측 발생 — forecast_id/시각 불일치 가능성"
print(submission.shape)
submission.head(3)

(8760, 5)


,forecast_id,forecast_kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,forecast_0001,2025-01-01 01:00:00,19779.438485,20056.442465,18675.908203
1,forecast_0002,2025-01-01 02:00:00,19639.124123,19972.376155,18521.154297
2,forecast_0003,2025-01-01 03:00:00,19072.933248,19558.803724,18707.765625


## 5. Sanity check — 예측이 그럴듯한가

`ensemble-final` 스킬 4절: 재학습 모델은 검증 점수가 없으므로, 예측 분포가 과거(2024) 패턴과 형태가 비슷한지로 명백한 오류만 확인한다. 2025년의 실제 날씨가 2024년과 정확히 같을 수는 없으니 **완전히 같은 모양일 필요는 없고, 계절 패턴(겨울↑여름↓ 등)이 뒤집히거나 극단적으로 어긋나지 않는지**를 본다.

In [6]:
train_features_for_compare = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")
train_2024 = train_features_for_compare[train_features_for_compare["forecast_kst_dtm"].dt.year == 2024].copy()
train_2024["month"] = train_2024["forecast_kst_dtm"].dt.month

submission_compare = submission.copy()
submission_compare["month"] = submission_compare["forecast_kst_dtm"].dt.month

monthly_2024 = train_2024.groupby("month")[TARGET_COLS].mean()
monthly_2025_pred = submission_compare.groupby("month")[TARGET_COLS].mean()

monthly_compare = monthly_2024.join(monthly_2025_pred, lsuffix="_actual_2024", rsuffix="_pred_2025")
print("월별 평균 발전량(kWh) — 2024년 실제 vs 2025년 예측:")
print(monthly_compare)

for g in TARGET_COLS:
    at_zero = (submission[g] <= 1e-6).mean()
    at_capacity = (submission[g] >= CAPACITY_KWH[g] - 1e-6).mean()
    print(f"{g}: 0kWh 근접 비율={at_zero:.1%}, 설비용량 근접 비율={at_capacity:.1%}")

월별 평균 발전량(kWh) — 2024년 실제 vs 2025년 예측:
       kpx_group_1_actual_2024  kpx_group_2_actual_2024  \
month                                                     
1                  8454.765524              8120.868776   
2                  4767.522322              5026.487644   
3                  7976.937009              7740.666558   
4                  4649.503486              4805.846233   
5                  7884.598289              7909.422819   
6                  3872.189119              4082.668246   
7                  9162.998454              9709.697989   
8                  3026.687923              3190.564832   
9                  2644.849539              3061.348313   
10                 3974.258054              3942.182497   
11                 5838.134021              5954.944110   
12                13039.120046             13987.720469   

       kpx_group_3_actual_2024  kpx_group_1_pred_2025  kpx_group_2_pred_2025  \
month                                                 

### 5-1. 월별 차이가 큰 이유 — 입력 풍속 자체가 다른지 확인

위 표를 보면 일부 달(2월·4월·6월·8월·9월·11월)은 예측이 2024년 실제보다 80~180%나 높고, 반대로 7월은 오히려 낮다. 이 정도로 큰 차이가 파이프라인 버그 때문인지, 아니면 "2025년 예보의 풍속 자체가 그 정도로 다르기 때문"인지 구분해야 한다. `wind-domain-features` 스킬 1절: **발전량은 풍속의 세제곱(v³)에 비례**하므로, 풍속이 조금만 달라도(예: 1.2배) 발전량 예측은 훨씬 크게(1.2³≈1.7배) 벌어질 수 있다 — 그러니 먼저 입력 풍속 자체를 연도별로 비교해서, 발전량 차이가 "풍속 차이의 자연스러운 증폭"으로 설명되는지 본다.

In [7]:
test_features_month = test_features.copy()
test_features_month["month"] = test_features_month["forecast_kst_dtm"].dt.month
train_features_for_compare["month"] = train_features_for_compare["forecast_kst_dtm"].dt.month
train_features_for_compare["year"] = train_features_for_compare["forecast_kst_dtm"].dt.year

wind_cols = ["gfs_ws100m", "kpx_group_1_ldaps_ws10m", "kpx_group_2_ldaps_ws10m", "kpx_group_3_ldaps_ws10m"]

wind_2024 = train_features_for_compare[train_features_for_compare["year"] == 2024].groupby("month")[wind_cols].mean()
wind_2025 = test_features_month.groupby("month")[wind_cols].mean()

wind_ratio = wind_2025[wind_cols] / wind_2024[wind_cols]
wind_ratio.columns = [f"{c}_2025/2024_ratio" for c in wind_cols]
print("풍속 비율(2025 예보 / 2024 실제, 1보다 크면 2025이 더 셈):")
wind_ratio

풍속 비율(2025 예보 / 2024 실제, 1보다 크면 2025이 더 셈):


,gfs_ws100m_2025/2024_ratio,kpx_group_1_ldaps_ws10m_2025/2024_ratio,kpx_group_2_ldaps_ws10m_2025/2024_ratio,kpx_group_3_ldaps_ws10m_2025/2024_ratio
month,,,,
1,1.416432,0.984764,0.998349,1.014259
2,1.803708,1.308924,1.399153,1.501640
3,0.951351,0.941501,0.942679,0.943522
4,1.868146,1.327292,1.346851,1.403293
5,1.019231,1.015951,0.985976,0.955227
6,1.329547,1.198119,1.196612,1.197879
7,0.586282,0.692582,0.706719,0.708369
8,1.253062,1.195015,1.262214,1.328352
9,1.170748,1.205745,1.194108,1.187842


## 6. 제출 파일 저장 + 검증

파일명 규칙(`CLAUDE.md` 8절): `submissions/YYYYMMDD_vN_설명.csv`. **`validate_submission()` 통과가 제출 전 필수 조건**이다. `description`은 이 예측의 핵심 설정을 요약해 나중에 파일 이름만 보고도 어떤 모델인지 알 수 있게 한다.

In [8]:
today_str = date.today().strftime("%Y%m%d")
version = "v2"
description = "catboost_mlp_groupblend_quantile070"
submission_path = SUBMISSIONS_DIR / f"{today_str}_{version}_{description}.csv"

submission.to_csv(submission_path, index=False, encoding="utf-8-sig")
print(f"저장: {submission_path}")

validate_submission(submission_path)

저장: d:\공모전\wind_forecast\submissions\20260723_v2_catboost_mlp_groupblend_quantile070.csv
[PASS] 20260723_v2_catboost_mlp_groupblend_quantile070.csv: 컬럼/행 수/ID 불변/결측·음수·상한 초과 없음 확인 완료 (8760행)


True

## 요약

- `train.ipynb`가 저장한 CatBoost seed 3개(42/7/2024) + MLP seed 3개(42/7/2024) 모델과 `model_config.json`을 불러와 2025년 테스트 기간을 예측했다
- 그룹별로 CatBoost·MLP 각각 seed 평균 앙상블 → 그룹별 블렌드 가중치(`BLEND_WEIGHTS`: group_1=0.4/group_2=0.5/group_3=1.0)로 섞음 → `[0, 설비용량]` 클리핑
- `sample_submission.csv`의 `forecast_id` 순서에 병합해서 순서 불일치를 원천 차단했다
- 2024년 월별 실제 발전량과 비교하는 sanity check로 명백한 오류가 없는지 확인했다
- `validate_submission()`을 통과한 제출 파일을 `submissions/`에 저장했다(`v2`, 설명 `catboost_mlp_groupblend_quantile070`) — **실제 대회 플랫폼 제출은 민석님이 직접 진행**하고, 일일 5회 제한을 고려해 신중하게 선택

**다음**: 실제 제출 후 리더보드 점수를 `HANDOFF.md`에 기록(`session-handoff` 스킬) — 첫 제출(CatBoost 단독, 0.62256)과 로컬 CV 개선폭(+0.0082)이 리더보드에서도 재현되는지가 이번 제출의 핵심 관전 포인트